# Практическое задание 2: Улучшение качества LLM классификатора

In [ ]:
%pip install -q -r requirements.txt


Файл с данными можно скачать по ссылке: https://disk.yandex.ru/d/ErrpfkgFkdtOIQ.  
Подгружаем данные:

In [ ]:
import pandas as pd

data = pd.read_csv("tweets_coronavirus_new.csv")

Для запуска задайте переменную окружения `MISTRAL_API_KEY`.
Имя переменной приведено в `.env.example`; сам ключ в ноутбук не записывается.


In [ ]:
import os
from mistralai.client import Mistral

llm_client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])


Посмотрим на наши данные:

In [ ]:
data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,8041,52993,"Livingston County, NY",18-03-2020,COVID-19 PRICE GOUGING HOTLINE: If you see exc...,Negative
1,22726,67678,"Adelaide, Australia",24-03-2020,URGENT PLEASE SHARE Hearing reports of fatalit...,Negative
2,36557,81509,"McCool, MS",07-04-2020,Taste of Power cuts prices with 50 while donat...,Positive
3,17518,62470,NaN,21-03-2020,Just been to supermarket to get a few things. ...,Negative
4,17470,62422,Morgantown WV,21-03-2020,@jacknicas Right...he had no idea that Covid-1...,Negative


In [ ]:
print(data.OriginalTweet[:5].to_markdown())

|    | OriginalTweet                                                                                                                                                                                                                                                                                                                                                                                       |
|---:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  0 | COVID-19 PRICE GOUGING HOTLINE: If you see excessively priced consumer goods and services that are used primarily for personal, family or household purposes to prevent or respond to the COVID-19 viru

In [ ]:
data.Sentiment.value_counts()

Sentiment
Negative    59
Positive    41
Name: count, dtype: int64

Наша задача &mdash; предсказывать сентимент твита. Закодируем в числа наш таргет:

In [ ]:
def text2label(text) -> int:
    if text in (1, "Positive", "Extremely Positive"):
        return 1
    if text in (0, "Negative", "Extremely Negative"):
        return 0
    raise ValueError(f"Неизвестная истинная метка: {text!r}")


In [ ]:
data["Sentiment"] = data["Sentiment"].apply(text2label)

In [ ]:
data.Sentiment.value_counts()

Sentiment
0    59
1    41
Name: count, dtype: int64

Определим модель, промпты и параметры генерации LLM:

In [ ]:
MODEL_NAME = "mistral-large-4"

SYSTEM_PROMPT = "Please classify the user's text. Respond with only one word: 'positive' or 'negative'. No other words allowed."

SAMPLING_PARAMETERS = {'temperature': 0.1, 'top_p': 1.0, 'reasoning_effort': 'low', 'max_tokens': 4096}

MAX_CONCURRENT = 3
REQUEST_DELAY = 1.3


Функция для обработки ответа модели. Здесь вы переводите ответ модели в метку класса (0 или 1):

In [ ]:
from mistralai.client.models import TextChunk


def response2label(response) -> int:
    if isinstance(response, list):
        response = "".join(
            chunk.text for chunk in response if isinstance(chunk, TextChunk)
        )
    if not isinstance(response, str):
        raise TypeError(f"Неожиданный тип ответа: {type(response)}")
    answer = response.strip().lower()
    labels = {"positive": 1, "negative": 0}
    if answer not in labels:
        raise ValueError(f"Неожиданный ответ: {answer!r}")
    return labels[answer]


Обрабатываем датасет: один твит на запрос, до трёх запросов одновременно.
`REQUEST_DELAY` задаёт общий интервал между стартами запросов для соблюдения лимита API.
Повторяются только временные ошибки API и соединения. Предсказания возвращаются
в порядке исходных твитов; при ошибке оставшиеся задачи отменяются.


In [ ]:
import asyncio
import httpx2 as httpx
from sklearn.metrics import accuracy_score
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_fixed
from tqdm.auto import tqdm

request_lock = asyncio.Lock()
next_request_start = 0.0


def is_temporary_error(error):
    status = getattr(error, "status_code", None)
    return (
        status == 429 or isinstance(status, int) and status >= 500
        or isinstance(error, (httpx.TransportError, TimeoutError))
    )


@retry(
    retry=retry_if_exception(is_temporary_error),
    stop=stop_after_attempt(3), wait=wait_fixed(20), reraise=True,
)
async def classify_sentiment(text):
    global next_request_start
    loop = asyncio.get_running_loop()
    async with request_lock:
        await asyncio.sleep(max(0, next_request_start - loop.time()))
        next_request_start = loop.time() + REQUEST_DELAY
    response = await llm_client.chat.complete_async(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        timeout_ms=180000,
        **SAMPLING_PARAMETERS,
    )
    return response2label(response.choices[0].message.content)


async def classify_batch(texts):
    texts = list(texts)
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    progress = tqdm(total=len(texts), desc="Твиты")

    async def classify_one(text):
        async with semaphore:
            label = await classify_sentiment(text)
            progress.update(1)
            return label

    tasks = [asyncio.create_task(classify_one(text)) for text in texts]
    try:
        return await asyncio.gather(*tasks)
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        progress.close()


In [ ]:
y_pred = await classify_batch(data["OriginalTweet"])
y_true = data["Sentiment"]

acc = accuracy_score(y_true, y_pred)
error_indices = [i for i, (actual, predicted) in enumerate(zip(y_true, y_pred))
                 if actual != predicted]
print("Accuracy:", acc)
print("error_indices:", error_indices)


**Выводы по экспериментам**

- Оставлены короткая исходная инструкция и один твит на запрос. Подробные
  контекстные инструкции дали 89–95%, инструкции о намерении автора — 83–86%.
  Усложнение промпта не принесло устойчивого улучшения.
- В ночной серии короткая инструкция с `temperature=0` и `reasoning_effort="low"`
  дала 95–98% в 23 прогонах со средним 96.78%, но с разными `random_seed`.
  Для `seed=29` есть только один результат 98%; повторяемость не подтверждена.
- Пакеты по 100 твитов дали 91–95%. Ансамбль из 11 голосов получил 98%, затем
  97% и занимал около 42 минут на 100 твитов. Одиночные запросы проще и быстрее.
- Сейчас проверяется `temperature=0.1` без фиксированного `random_seed`.
  Параллельность — 3, общий интервал между стартами — 1.3 секунды.
  Устойчивое повышение accuracy на этой настройке пока не подтверждено.

Все 100 твитов использовались для настройки. Качество на новых данных не проверено.
При сравнении вариантов нужно учитывать повторные прогоны, а не только лучший.
